## 09 — Building density / size vs F1 analysis

Examines whether reference building density and mean building size predict
per-tile F1 accuracy for each candidate dataset.

**Input:** `outputs/scratch/per_tile_enriched_all_cities.csv`  
**Outputs:** figures in `outputs/figures/`, summary statistics printed inline.

**Statistical approach:**
- Spearman ρ (marginal; city-level confounding not removed)
- Kruskal–Wallis + Dunn post-hoc (Bonferroni) across density quartile groups
- Linear mixed-effects model: city as random intercept, density + size + dataset as fixed effects

In [ ]:
!pip install -q scikit-posthocs statsmodels

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 1 — Setup and load ───────────────────────────────────────────────────
import warnings; warnings.filterwarnings('ignore')
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.formula.api as smf
import scikit_posthocs as sp
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import yaml

CONFIG_PATH  = Path('/content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/configs/validation_configs.yaml')
PROJECT_ROOT = CONFIG_PATH.parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'
SCRATCH_DIR = PROJECT_ROOT / 'outputs' / 'scratch'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = SCRATCH_DIR / 'per_tile_enriched_all_cities.csv'
df = pd.read_csv(CSV_PATH)

# Normalise dataset name capitalisation for display
DS_LABELS = {'overture': 'Overture', 'gba': 'GBA', 'globfp': 'GlobFP'}
df['dataset_label'] = df['dataset'].str.lower().map(DS_LABELS).fillna(df['dataset'])

DENSITY_COL = 'ref_building_density_per_km2'
SIZE_COL    = 'mean_ref_building_area_m2'
F1_COL      = 'f1'

# ── Data quality summary ──────────────────────────────────────────────────────
nan_density = df[DENSITY_COL].isna().mean() * 100
nan_f1      = df[F1_COL].isna().mean() * 100
nan_size    = df[SIZE_COL].isna().mean() * 100

print('=== Dataset overview ===')
print(f'  Total rows          : {len(df):,}')
print(f'  Unique cities       : {df["city"].nunique()}')
print(f'  Unique datasets     : {df["dataset"].unique().tolist()}')
print(f'  Density range       : {df[DENSITY_COL].min():.1f} – {df[DENSITY_COL].max():.1f} bldg/km²')
print(f'  NaN in density      : {nan_density:.1f}%')
print(f'  NaN in f1           : {nan_f1:.1f}%')
print(f'  NaN in mean size    : {nan_size:.1f}%')

# Drop rows missing either predictor or outcome
df_clean = df.dropna(subset=[DENSITY_COL, SIZE_COL, F1_COL]).copy()
print(f'\n  Rows after dropping NaNs : {len(df_clean):,} ({len(df_clean)/len(df)*100:.1f}% retained)')

In [ ]:
# ── Cell 2 — Spearman correlation ────────────────────────────────────────────
# Note: tiles are nested within cities. This is a marginal (pooled) correlation;
# city-level confounding is removed in the mixed-effects model (Cell 6).

print('=== Spearman ρ — marginal correlation (all tiles pooled within dataset) ===')
print('NOTE: tiles are nested within cities; city confounding not yet removed.\n')

spearman_results = []
datasets = sorted(df_clean['dataset'].str.lower().unique())

for ds in datasets:
    sub = df_clean[df_clean['dataset'].str.lower() == ds]
    label = DS_LABELS.get(ds, ds)

    rho_d, p_d = stats.spearmanr(sub[DENSITY_COL], sub[F1_COL])
    rho_s, p_s = stats.spearmanr(sub[SIZE_COL],    sub[F1_COL])

    spearman_results.append({
        'dataset': label, 'n_tiles': len(sub),
        'rho_density': rho_d, 'p_density': p_d,
        'rho_size':    rho_s, 'p_size':    p_s,
    })

    sig_d = '***' if p_d < 0.001 else ('**' if p_d < 0.01 else ('*' if p_d < 0.05 else 'ns'))
    sig_s = '***' if p_s < 0.001 else ('**' if p_s < 0.01 else ('*' if p_s < 0.05 else 'ns'))
    print(f'{label} (n={len(sub):,})')
    print(f'  density vs F1 : ρ = {rho_d:+.3f}  p = {p_d:.2e}  {sig_d}')
    print(f'  size    vs F1 : ρ = {rho_s:+.3f}  p = {p_s:.2e}  {sig_s}')
    print()

spearman_df = pd.DataFrame(spearman_results)
print('(* p<0.05  ** p<0.01  *** p<0.001  ns = not significant)')

In [ ]:
# ── Cell 3 — Density quartile groups + Kruskal–Wallis + Dunn post-hoc ────────
# Quartiles based on global distribution across all tiles and datasets.

quantiles = df_clean[DENSITY_COL].quantile([0, 0.25, 0.5, 0.75, 1.0]).values
q_labels  = [
    f'Q1\n(<{quantiles[1]:.0f})',
    f'Q2\n({quantiles[1]:.0f}–{quantiles[2]:.0f})',
    f'Q3\n({quantiles[2]:.0f}–{quantiles[3]:.0f})',
    f'Q4\n(>{quantiles[3]:.0f})',
]
df_clean['density_q'] = pd.cut(
    df_clean[DENSITY_COL],
    bins=quantiles,
    labels=q_labels,
    include_lowest=True,
)

print('=== Density quartile boundaries (global, all tiles × datasets) ===')
for i, (lo, hi) in enumerate(zip(quantiles[:-1], quantiles[1:])):
    n = (df_clean['density_q'] == q_labels[i]).sum()
    print(f'  Q{i+1}: {lo:.1f} – {hi:.1f} bldg/km²  ({n:,} tiles)')

print()
print('=== Kruskal–Wallis test: F1 across density quartiles ===')
kw_results = []

for ds in datasets:
    label = DS_LABELS.get(ds, ds)
    sub = df_clean[df_clean['dataset'].str.lower() == ds].dropna(subset=['density_q'])
    groups = [grp[F1_COL].values for _, grp in sub.groupby('density_q', observed=True)]
    groups = [g for g in groups if len(g) > 0]

    if len(groups) < 2:
        print(f'{label}: insufficient groups — skipping')
        continue

    stat, p = stats.kruskal(*groups)
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    kw_results.append({'dataset': label, 'H': stat, 'p': p, 'sig': sig})
    print(f'{label}: H = {stat:.2f}  p = {p:.2e}  {sig}')

    if p < 0.05:
        print(f'  → Dunn post-hoc (Bonferroni):')
        dunn = sp.posthoc_dunn(
            sub, val_col=F1_COL, group_col='density_q', p_adjust='bonferroni'
        )
        print(dunn.round(4).to_string())
    print()

print('(* p<0.05  ** p<0.01  *** p<0.001  ns = not significant)')

In [ ]:
# ── Cell 4 — Box plot: F1 by density quartile, faceted by dataset ─────────────
# CVD-safe neutral palette — one hue per dataset panel, quartile on x-axis.
# Okabe-Ito categorical colours (3 of 8 slots):
#   Overture #0072B2  GBA #E69F00  GlobFP #009E73

DS_COLORS = {'Overture': '#0072B2', 'GBA': '#E69F00', 'GlobFP': '#009E73'}
ds_order  = ['Overture', 'GBA', 'GlobFP']
ds_order  = [d for d in ds_order if d in df_clean['dataset_label'].unique()]

fig, axes = plt.subplots(1, len(ds_order), figsize=(4.5 * len(ds_order), 5), sharey=True)
if len(ds_order) == 1:
    axes = [axes]

for ax, ds_label in zip(axes, ds_order):
    sub = df_clean[df_clean['dataset_label'] == ds_label].dropna(subset=['density_q'])
    color = DS_COLORS.get(ds_label, '#666666')

    sns.boxplot(
        data=sub, x='density_q', y=F1_COL, ax=ax,
        order=q_labels,
        color=color, width=0.55, linewidth=1.2,
        flierprops=dict(marker='o', markersize=2, alpha=0.25,
                        markerfacecolor=color, markeredgecolor='none'),
        medianprops=dict(color='white', linewidth=2),
    )

    # Annotate n per group
    for i, ql in enumerate(q_labels):
        n = (sub['density_q'] == ql).sum()
        ax.text(i, -0.06, f'n={n:,}', ha='center', va='top',
                fontsize=7.5, color='#555555', transform=ax.get_xaxis_transform())

    ax.set_title(ds_label, fontsize=12, fontweight='bold', color=color, pad=8)
    ax.set_xlabel('Density quartile (bldg/km²)', fontsize=9)
    ax.set_ylabel('F1 score' if ax == axes[0] else '', fontsize=9)
    ax.set_ylim(-0.05, 1.05)
    ax.yaxis.set_major_locator(mticker.MultipleLocator(0.2))
    ax.grid(axis='y', color='#e0e0e0', linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)
    sns.despine(ax=ax, left=False)

fig.suptitle('F1 score by reference building density quartile', fontsize=13, y=1.01)
fig.tight_layout()

out_path = FIGURES_DIR / 'f1_by_density_quartile.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out_path}')

In [ ]:
# ── Cell 5 — Scatter plots ─────────────────────────────────────────────────────
# (a) density vs F1   (b) mean building size vs F1
# Coloured by dataset (same Okabe-Ito palette as Cell 4). Log-scale x for density.

def _scatter(ax, x_col, x_label, log_x=False):
    for ds_label in ds_order:
        sub = df_clean[df_clean['dataset_label'] == ds_label]
        ax.scatter(
            sub[x_col], sub[F1_COL],
            color=DS_COLORS.get(ds_label, '#666'), alpha=0.3,
            s=8, linewidths=0, label=ds_label, rasterized=True,
        )
    if log_x:
        ax.set_xscale('log')
    ax.set_xlabel(x_label, fontsize=10)
    ax.set_ylabel('F1 score', fontsize=10)
    ax.set_ylim(-0.02, 1.05)
    ax.grid(color='#e8e8e8', linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)
    ax.legend(title='Dataset', fontsize=8, title_fontsize=8,
              markerscale=3, framealpha=0.85)
    sns.despine(ax=ax)

# (a) Density vs F1
fig_a, ax_a = plt.subplots(figsize=(7, 5))
_scatter(ax_a, DENSITY_COL,
         'Reference building density (bldg/km²)', log_x=True)
ax_a.set_title('Reference building density vs F1', fontsize=12)
fig_a.tight_layout()
path_a = FIGURES_DIR / 'density_f1_scatter.png'
fig_a.savefig(path_a, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {path_a}')

# (b) Mean building size vs F1
fig_b, ax_b = plt.subplots(figsize=(7, 5))
_scatter(ax_b, SIZE_COL,
         'Mean reference building area (m²)')
ax_b.set_title('Mean reference building size vs F1', fontsize=12)
fig_b.tight_layout()
path_b = FIGURES_DIR / 'size_f1_scatter.png'
fig_b.savefig(path_b, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {path_b}')

In [ ]:
# ── Cell 6 — Linear mixed-effects model ──────────────────────────────────────
# f1 ~ density + mean_size + C(dataset)  |  random intercept: city
#
# Density is standardised (z-score) for numerical stability and
# interpretability (coefficient = change in F1 per 1-SD increase in density).

df_lme = df_clean[[F1_COL, DENSITY_COL, SIZE_COL, 'dataset', 'city']].dropna().copy()

df_lme['density_z'] = (df_lme[DENSITY_COL] - df_lme[DENSITY_COL].mean()) / df_lme[DENSITY_COL].std()
df_lme['size_z']    = (df_lme[SIZE_COL]    - df_lme[SIZE_COL].mean())    / df_lme[SIZE_COL].std()
df_lme['dataset']   = df_lme['dataset'].str.lower()

formula = 'f1 ~ density_z + size_z + C(dataset)'

model  = smf.mixedlm(formula, data=df_lme, groups=df_lme['city'])
result = model.fit(reml=True)

print(result.summary())
print()
print('=== Key fixed-effect coefficients ===')

for name in ['density_z', 'size_z']:
    coef = result.params[name]
    pval = result.pvalues[name]
    ci   = result.conf_int().loc[name]
    sig  = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else 'ns'))
    label = 'density (z)' if name == 'density_z' else 'mean size (z)'
    print(f'  {label:<20}: β = {coef:+.4f}  95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}]  p = {pval:.2e}  {sig}')

print()
print('Interpretation: β is change in F1 per 1-SD increase in the predictor,'
      ' holding other variables constant and accounting for city-level variation.')
print(f'Random-effect variance (city): {result.cov_re.iloc[0,0]:.4f}')

In [ ]:
# ── Cell 7 — Limitation note & final summary ──────────────────────────────────

# Geographic coverage from city slugs
iso_region = {
    'afg': 'South Asia', 'bgd': 'South Asia', 'ind': 'South Asia', 'yem': 'Middle East',
    'chn': 'East Asia', 'jpn': 'East Asia', 'kor': 'East Asia',
    'are': 'Middle East', 'kwt': 'Middle East', 'sau': 'Middle East',
    'alg': 'North Africa', 'egy': 'North Africa', 'lby': 'North Africa',
    'sdn': 'North Africa', 'ssd': 'Sub-Saharan Africa',
    'gha': 'Sub-Saharan Africa', 'sen': 'Sub-Saharan Africa',
    'uga': 'Sub-Saharan Africa', 'lbr': 'Sub-Saharan Africa',
    'nga': 'Sub-Saharan Africa', 'sle': 'Sub-Saharan Africa',
    'mwi': 'Sub-Saharan Africa', 'moz': 'Sub-Saharan Africa',
    'zmb': 'Sub-Saharan Africa', 'swz': 'Sub-Saharan Africa',
    'zaf': 'Sub-Saharan Africa', 'ner': 'Sub-Saharan Africa',
    'ken': 'Sub-Saharan Africa', 'tjk': 'Central Asia', 'uzb': 'Central Asia',
    'mmr': 'South-East Asia', 'phl': 'South-East Asia',
    'bra': 'Latin America', 'chl': 'Latin America', 'col': 'Latin America',
    'mex': 'Latin America', 'pan': 'Latin America', 'per': 'Latin America',
    'ant': 'Caribbean', 'blz': 'Caribbean', 'cvg': 'Caribbean',
    'dom': 'Caribbean', 'grd': 'Caribbean', 'jam': 'Caribbean',
    'lca': 'Caribbean', 'maf': 'Caribbean', 'sxm': 'Caribbean',
    'tto': 'Caribbean', 'ton': 'Pacific',
    'aus': 'Oceania',
    'gbr': 'Europe', 'nld': 'Europe', 'rou': 'Europe', 'rus': 'Europe', 'ukr': 'Europe',
    'usa': 'North America',
}

included_cities = df_clean['city'].unique()
regions = sorted(set(
    iso_region.get(c[:3].lower(), 'Other') for c in included_cities
))

d_min = df_clean[DENSITY_COL].min()
d_max = df_clean[DENSITY_COL].max()

print('=== Analysis limitations & coverage ===')
print(f'  Cities included   : {len(included_cities)} of 136 total')
print(f'  Cities excluded   : ~65 (missing AOI/tiles GPKG — data cleaned from Drive after pipeline run)')
print(f'  Tiles analysed    : {len(df_clean):,}')
print(f'  Density range     : {d_min:.1f} – {d_max:.1f} bldg/km²')
print(f'  Regions covered   : {", ".join(regions)}')
print()
print('  Statistical caveats:')
print('  - Spearman ρ (Cell 2) is marginal: tiles are NOT independent (nested in cities).')
print('    City-level confounding inflates/deflates ρ. Treat as exploratory only.')
print('  - LME (Cell 6) accounts for city nesting via random intercept.')
print('    Density coefficient is interpretable as within-city effect.')
print('  - 65 excluded cities may introduce geographic bias if the missing')
print('    cities are systematically different in density or building type.')

# Save the analysis input for reproducibility
out_csv = SCRATCH_DIR / 'density_analysis_input.csv'
df_clean.to_csv(out_csv, index=False)
print(f'\nAnalysis input saved → {out_csv}')

# ── Final summary ─────────────────────────────────────────────────────────────
print()
print('=== Result summary ===')
print()
print('Spearman ρ (density vs F1):')
for _, row in spearman_df.iterrows():
    sig = '***' if row.p_density < 0.001 else ('**' if row.p_density < 0.01 else ('*' if row.p_density < 0.05 else 'ns'))
    print(f'  {row.dataset:<10}: ρ = {row.rho_density:+.3f}  p = {row.p_density:.2e}  {sig}')

print()
print('Kruskal–Wallis p-values (density quartiles vs F1):')
for r in kw_results:
    print(f'  {r["dataset"]:<10}: H = {r["H"]:.2f}  p = {r["p"]:.2e}  {r["sig"]}')

print()
print('LME fixed effects (city random intercept):')
for name, label in [('density_z', 'density (z)'), ('size_z', 'mean size (z)')]:
    coef = result.params[name]
    pval = result.pvalues[name]
    sig  = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else 'ns'))
    print(f'  {label:<20}: β = {coef:+.4f}  p = {pval:.2e}  {sig}')

print()
print('Figure paths:')
print(f'  {FIGURES_DIR / "f1_by_density_quartile.png"}')
print(f'  {FIGURES_DIR / "density_f1_scatter.png"}')
print(f'  {FIGURES_DIR / "size_f1_scatter.png"}')